In [1]:
import mysql.connector
import pandas as pd

conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345",
    database="cust_db"
)
print("Connected to database successfully!")

query="select * from customer_shopping_behavior"

df=pd.read_sql(query, conn)

Connected to database successfully!


C:\Users\Aarya Khire\AppData\Local\Temp\ipykernel_18856\2611798999.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query, conn)


In [2]:
df.info()
df.isnull().sum()
df.duplicated().sum()
df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3863 entries, 0 to 3862
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cust_id                 3863 non-null   int64  
 1   Age                     3863 non-null   int64  
 2   Gender                  3863 non-null   object 
 3   Item Purchased          3863 non-null   object 
 4   Category                3863 non-null   object 
 5   Purchase Amount (USD)   3863 non-null   int64  
 6   Location                3863 non-null   object 
 7   Size                    3863 non-null   object 
 8   Color                   3863 non-null   object 
 9   Season                  3863 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3863 non-null   object 
 12  Shipping Type           3863 non-null   object 
 13  Discount Applied        3863 non-null   object 
 14  Promo Code Used         3863 non-null   

,cust_id,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3863.000000,3863.000000,3863,3863,3863,3863.000000,3863,3863,3863,3863,3863.000000,3863,3863,3863,3863,3863.000000,3863,3863
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Jewelry,Clothing,NaN,California,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2615,170,1718,NaN,95,1738,176,987,NaN,2830,667,2223,2223,NaN,673,577
mean,1961.465441,44.067564,NaN,NaN,NaN,59.702045,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.340668,NaN,NaN
std,1124.931676,15.207314,NaN,NaN,NaN,23.688325,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.453498,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,986.500000,31.000000,NaN,NaN,NaN,38.500000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1969.000000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2934.500000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [4]:
import pandas as pd
import numpy as np

df.columns = df.columns.str.strip().str.replace("ï»¿", "", regex=False)

df = df.rename(columns={
    "Purchase Amount (USD)": "purchase_amount_usd",
    "Review Rating": "review_rating",
    "Previous Purchases": "previous_purchases",
    "Item Purchased": "item_purchased",
    "Shipping Type": "shipping_type",
    "Payment Method": "payment_method",
    "Frequency of Purchases": "frequency_of_purchases",
    "Discount Applied": "discount_applied",
    "Promo Code Used": "promo_code_used",
    "Subscription Status": "subscription_status"
})

df["purchase_amount_usd"] = pd.to_numeric(df["purchase_amount_usd"], errors="coerce")
df["review_rating"] = pd.to_numeric(df["review_rating"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["previous_purchases"] = pd.to_numeric(df["previous_purchases"], errors="coerce")

In [5]:
bins = [0, 18, 25, 35, 45, 55, 100]
labels = ["0-18", "19-25", "26-35", "36-45", "46-55", "56+"]
df["age_group"] = pd.cut(df["Age"], bins=bins, labels=labels, right=True)

freq_map = {
    "Weekly": 4,
    "Fortnightly": 2,
    "Bi-Weekly": 2,
    "Monthly": 1,
    "Quarterly": 0.33,
    "Every 3 Months": 0.33,
    "Annually": 0.08
}
df["purchase_frequency_score"] = df["frequency_of_purchases"].map(freq_map)

In [8]:
# Categories generating the most revenue
category_revenue = (
    df.groupby("Category", as_index=False)["purchase_amount_usd"]
    .sum()
    .sort_values("purchase_amount_usd", ascending=False)
)
print("Categories generating the most revenue:")
print(category_revenue)

Categories generating the most revenue:
      Category  purchase_amount_usd
1     Clothing               102968
0  Accessories                73480
2     Footwear                35778
3    Outerwear                18403


In [9]:
#. Items with the highest average rating
top_items_rating = (
    df.groupby("item_purchased", as_index=False)["review_rating"]
    .mean()
    .sort_values("review_rating", ascending=False)
)
print("\nItems with the highest average rating:")
print(top_items_rating)


Items with the highest average rating:
   item_purchased  review_rating
6          Gloves       3.862774
14        Sandals       3.844654
3           Boots       3.818881
8             Hat       3.801307
19          Skirt       3.785350
24        T-shirt       3.779310
7         Handbag       3.775163
1            Belt       3.763522
10         Jacket       3.763190
23        Sweater       3.760870
12        Jewelry       3.758824
20       Sneakers       3.757931
21          Socks       3.755063
0        Backpack       3.752448
22     Sunglasses       3.751572
5           Dress       3.750000
17          Shoes       3.749660
4            Coat       3.727673
13          Pants       3.720833
9          Hoodie       3.716779
18         Shorts       3.714194
15          Scarf       3.705806
2          Blouse       3.680473
11          Jeans       3.648387
16          Shirt       3.622024


In [10]:
#Age groups that spend more
age_spend = (
    df.groupby("age_group", as_index=False)["purchase_amount_usd"]
    .sum()
    .sort_values("purchase_amount_usd", ascending=False)
)
print("\nAge groups that spend more:",age_spend)


Age groups that spend more:   age_group  purchase_amount_usd
5       56+                64560
4     46-55                44927
2     26-35                44008
3     36-45                42853
1     19-25                30142
0      0-18                 4139


C:\Users\Aarya Khire\AppData\Local\Temp\ipykernel_18856\3502072064.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("age_group", as_index=False)["purchase_amount_usd"]


In [11]:
# Do discounts increase purchase amount?
discount_analysis = (
    df.groupby("discount_applied")["purchase_amount_usd"]
    .agg(["mean", "median", "count", "sum"])
    .sort_values("mean", ascending=False)
)
print("\nDiscount impact on purchase amount:")
print(discount_analysis)


Discount impact on purchase amount:
                       mean  median  count     sum
discount_applied                                  
No                60.130454    60.0   2223  133670
Yes               59.121341    59.0   1640   96959


In [12]:
# Most common shipping types
shipping_common = df["shipping_type"].value_counts().reset_index()
shipping_common.columns = ["shipping_type", "count"]
print("\nMost common shipping types:")
print(shipping_common)


Most common shipping types:
    shipping_type  count
0   Free Shipping    667
1        Standard    650
2    Next Day Air    645
3    Store Pickup    645
4         Express    636
5  2-Day Shipping    620


In [ ]:
# Most common payment methods
payment_common = df["payment_method"].value_counts().reset_index()
payment_common.columns = ["payment_method", "count"]
print("\nMost common payment methods:")
print(payment_common)


Most common payment methods:
  payment_method  count
0         PayPal    673
1           Cash    666
2    Credit Card    662
3     Debit Card    630
4          Venmo    626
5  Bank Transfer    606


In [14]:
# Sales by location
location_sales = (
    df.groupby("Location", as_index=False)["purchase_amount_usd"]
    .sum()
    .sort_values("purchase_amount_usd", ascending=False))
print("\nSales by location:")
print(location_sales)


Sales by location:
          Location  purchase_amount_usd
25         Montana                 5717
4       California                 5605
12        Illinois                 5537
11           Idaho                 5438
27          Nevada                 5379
0          Alabama                 5261
26        Nebraska                 5172
33    North Dakota                 5169
31        New York                 5136
47   West Virginia                 5045
30      New Mexico                 5014
22       Minnesota                 4977
37    Pennsylvania                 4926
23     Mississippi                 4883
1           Alaska                 4867
17       Louisiana                 4800
44         Vermont                 4791
41       Tennessee                 4772
19        Maryland                 4767
32  North Carolina                 4742
3         Arkansas                 4731
7         Delaware                 4685
42           Texas                 4662
13         Indiana  

In [16]:
#Are high-frequency buyers also high spenders?
freq_corr = df[["purchase_frequency_score", "purchase_amount_usd"]].corr().iloc[0, 1]

freq_spend = (
    df.groupby("frequency_of_purchases", as_index=False)["purchase_amount_usd"]
    .agg(["mean", "sum", "count"])
    .sort_values("mean", ascending=False)
)
print("\nCorrelation between purchase frequency and amount spent:", freq_corr)
print("\nAverage spend by purchase frequency:")
print(freq_spend)


Correlation between purchase frequency and amount spent: -0.013944483382445765

Average spend by purchase frequency:
  frequency_of_purchases       mean    sum  count
1              Bi-Weekly  60.747232  32925    542
0               Annually  60.072566  33941    565
5              Quarterly  60.019749  33431    557
2         Every 3 Months  59.908146  34567    577
4                Monthly  59.295620  32494    548
3            Fortnightly  59.022263  31813    539
6                 Weekly  58.800000  31458    535


In [17]:
# Categories with strong sales but weak ratings
category_quality = (
    df.groupby("Category", as_index=False)
      .agg(total_sales=("purchase_amount_usd", "sum"),
           avg_rating=("review_rating", "mean"))
      .sort_values("total_sales", ascending=False)
)

weak_rating_strong_sales = category_quality.sort_values(
    ["total_sales", "avg_rating"], ascending=[False, True]
)
print("\nCategories with strong sales but weak ratings:")
print(weak_rating_strong_sales)


Categories with strong sales but weak ratings:
      Category  total_sales  avg_rating
1     Clothing       102968    3.721537
0  Accessories        73480    3.769976
2     Footwear        35778    3.793771
3    Outerwear        18403    3.745652


In [18]:

conn.close()